# HateMirage @ ICON 2026 - 2xT4 QLoRA (dual-GPU, HF-sync, resume, auto-eval)

**Tasks:** A=Target span - B=Intent+Implication generation - **Metrics:** SBERT-cosine + ROUGE-L F1 (greedy T=0)  
**Setup:** Kaggle accelerator `GPU T4 x2` - Internet ON - Secrets: `HF_TOKEN` (write) - Optional: `WANDB_API_KEY`  
**Flow:** env-check -> install -> HF login+config -> data -> RAG -> torchrun-DDP train (both T4s) -> Hub push -> inference -> eval -> Codabench zips.  
If the session hits the 9h limit or converges early, training saves + pushes and the notebook jumps straight to eval - never lose a run. Resume anytime from `HF_REPO`.

Paper: https://arxiv.org/abs/2603.02684 - Task: https://sai-kartheek-reddy.github.io/HateMirage-ICON2026/ - Codabench: https://www.codabench.org/competitions/17783/

In [ ]:
# CELL 1 - env check: must see 2x Tesla T4
import torch, os, time, shutil
T0 = time.time()
OUT = '/kaggle/working/hatemirage_out'
os.makedirs(OUT, exist_ok=True)
os.environ["OUT"] = OUT
print('GPUs:', torch.cuda.device_count(), '| OUT:', OUT)

In [ ]:
# CELL 2 - deps
!pip install -q -U "transformers>=4.44" accelerate peft trl bitsandbytes "datasets>=2.20" sentence-transformers rouge-score faiss-cpu huggingface_hub pandas scikit-learn openpyxl
import transformers
print('transformers', transformers.__version__)

In [ ]:
# CELL 3 - secrets + run config (HF_TOKEN_HARDCODED redacted -> hf_YOUR_TOKEN_HERE in repo copy)
import os
HF_TOKEN_HARDCODED = "hf_YOUR_TOKEN_HERE"
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = HF_TOKEN_HARDCODED
os.environ['HF_TOKEN'] = HF_TOKEN
HAS_HF = bool(HF_TOKEN)
HF_USER = 'blackpanter_252'
MODEL_ID = 'microsoft/Phi-3-mini-128k-instruct'
SMOKE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
EFFECTIVE_MODEL = MODEL_ID if HAS_HF else SMOKE_MODEL
HF_REPO = f'{HF_USER}/hatemirage-phi3-qlora'
WITH_RAG, MAX_RUNTIME_H = True, 8.2
EPOCHS, BS, GRAD_ACCUM, LORA_R = (3, 2, 8, 64) if HAS_HF else (1, 2, 2, 8)


In [ ]:
# CELL 4 - official task data (Development + Evaluation phases, pinned SHAs)
# Train.xlsx 04cb07ab | Val-for-dev a81fe96d | Val-Data-with-labels 838c3f83 | test-data-for-eval d76f741f
import glob, pandas as pd, os, urllib.request
GH = 'https://raw.githubusercontent.com/Sai-Kartheek-Reddy/HateMirage-ICON2026/main/'
OFFICIAL = {'train': 'Development%20Phase/Train.xlsx', 'devval': 'Development%20Phase/Val-for-dev.xlsx', 'vallab': 'Evaluation%20Phase/Val-Data-with-labels.xlsx', 'test': 'Evaluation%20Phase/test-data-for-eval.xlsx'}


In [ ]:
# CELL 5 - RAG index (FAISS, all-mpnet-base-v2, top-5)


In [ ]:
# CELL 6a - training config: torchrun DDP on BOTH T4s, Hub fingerprint resume (trainer_state.json + RESUME_INFO match), warm_adapter weights-only start


In [ ]:
%%writefile src/train_ddp.py
# full trainer: see src/train_ddp.py in this repo (identical)


In [ ]:
%%bash
torchrun --nproc_per_node=2 --master_port=29511 src/train_ddp.py --config $OUT/train_config.json 2>&1 | tee $OUT/ddp.log


In [ ]:
# CELL 7 - Hub push (lean, skips optimizer.pt) + inference load (balanced shard, native code, local-first adapter)


In [ ]:
# CELL 8 - inference (v2 official prompt: ## Comment/## Task/## Field, greedy, caps 96, To/Could norm, row fallbacks)
# CELL 9 - eval (MiniLM SBERT + ROUGE-L) + plots + Hub backup
# CELL 10 - official submissions (submission_official.xlsx + taskA/taskB csvs + zips)
# CELL 11 - resume playbook
